# The patent application title table (TLS202_APPLN_TITLE)

Welcome to the second table in PATSTAT, namely the TLS202_APPLN_TITLE table. It comprises three attributes: the first containts the application ID, the second the title of the application and the third the language of the title.

Let's start creating the PATSTAT client and accessing ORM. Then we import the TLS202_APPLN_TITLE table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS202_APPLN_TITLE

## APPLN_ID (Primary Key)

As seen in table TLS201, this is the unique identifier for each patent application in the PATSTAT database. We can use it to join this table with table TLS201.

In [2]:
# Import table TLS201
from epo.tipdata.patstat.database.models import TLS201_APPLN

appln_id = db.query(
    TLS202_APPLN_TITLE.appln_id,
    TLS201_APPLN.appln_nr
).join(
    TLS201_APPLN, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id
).limit(20000)

appln_id_df = patstat.df(appln_id)
appln_id_df

,appln_id,appln_nr
0,9177342,21006478
1,49228989,27627807
2,519457000,201880010591
3,413238654,201313966003
4,618400125,24202281
...,...,...
19995,550416316,202021642455
19996,40890774,20057001957
19997,18996192,0002019
19998,530059378,201880059047


## APPLN_TITLE_LG
This attribute contains the language of the title of the application.

Let's see which language is most commonly used to file applications.

In [7]:
from sqlalchemy import func

lg = db.query(
    TLS202_APPLN_TITLE.appln_title_lg,
    func.count(TLS202_APPLN_TITLE.appln_id).label('total_applications')
).group_by(
    TLS202_APPLN_TITLE.appln_title_lg
).order_by(
    func.count(TLS202_APPLN_TITLE.appln_id).desc()
).limit(10)

lg_df = patstat.df(lg)
lg_df

,appln_title_lg,total_applications
0,en,100090005
1,de,6258738
2,fr,2231980
3,es,1222210
4,ja,1120506
5,zh,1019313
6,pt,965643
7,ko,735099
8,it,721074
9,ru,522545


As we could expect, the most common language is English.

Suppose that we are also interested in knowing which application authorities have the most distinct languages used for the titles of the applications filed therein.

In [4]:
lg_auth = db.query(
    TLS201_APPLN.appln_auth,
    func.count(TLS202_APPLN_TITLE.appln_title_lg.distinct()).label('num_of_languages')  # Count the distinct number of title languages
).join(
    TLS201_APPLN, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id
).group_by(
    TLS201_APPLN.appln_auth
).order_by(
    func.count(TLS202_APPLN_TITLE.appln_title_lg.distinct()).desc()  
)

lg_auth_df = patstat.df(lg_auth)
lg_auth_df

,appln_auth,num_of_languages
0,PT,10
1,GR,6
2,WO,6
3,SE,6
4,BE,6
...,...,...
101,PE,1
102,MW,1
103,CG,1
104,TH,1


Out of curiosity, we can check which title languages are present among the applications filed at the EPO and the WIPO. We need to consider distinct combinations of `appln_auth`–`appln_title_lg` rows. However, we are not applying this to an aggregate function. In this case, we have to use the `distinct` operator.

In [5]:
# Import distinct from sqlalchemy
from sqlalchemy import distinct

lg_wo_ep = db.query(
    TLS201_APPLN.appln_auth,
    TLS202_APPLN_TITLE.appln_title_lg.label('distinct_languages')
).distinct(  # Consider distinct appln_auth-appln_title_lg rows combinations only
).join(
    TLS201_APPLN, TLS202_APPLN_TITLE.appln_id == TLS201_APPLN.appln_id
).filter(
    (TLS201_APPLN.appln_auth == 'WO') | (TLS201_APPLN.appln_auth == 'EP')
).order_by(
    TLS201_APPLN.appln_auth
)

lg_wo_ep_df = patstat.df(lg_wo_ep)
lg_wo_ep_df

,appln_auth,distinct_languages
0,EP,en
1,EP,de
2,WO,fr
3,WO,en
4,WO,el
5,WO,de
6,WO,es
7,WO,zh


## APPLN_TITLE

This attribute contains the title of the application. Only one of possibly multiple titles is stored.

We can take a look to the titles stored in the table

In [6]:
title = db.query(
    TLS202_APPLN_TITLE.appln_id,
    TLS202_APPLN_TITLE.appln_title_lg,
    TLS202_APPLN_TITLE.appln_title
).limit(20000)

title_df = patstat.df(title)
title_df

,appln_id,appln_title_lg,appln_title
0,23776793,el,ΣΥΣΤΗΜΑ ΕΜΦΑΝΙΣΕΩΣ ΔΙΑΛΕΙΠΟΥΣΗΣ ΦΩΤΕΙΝΗΣ ΔΙΑΦΗ...
1,607935429,uk,АНАЛОЙ
2,41817601,nl,NIEUW CHEMISCH OPBLAASMIDDEL.
3,411142491,ru,БУРОВОЕ ДОЛОТО
4,42874864,ru,"ЭЛЕКТРОННОЕ, ДИСТАНЦИОННОЕ УСТРОЙСТВО УПРАВЛЕН..."
...,...,...,...
19995,622788150,zh,一种烟机及其控制方法
19996,623941236,zh,一种高压微胶囊包填机
19997,625911682,zh,一种拼接型铝基板
19998,622650156,zh,一种建筑工程用照明设备
